# Домашнее задание: Занятие 33

**Тема: Батчи и эпохи -- обработка больших данных**

В этом задании вы закрепите понимание батчей, эпох и итераций. Каждое задание ссылается на конкретный слайд из презентации -- если забыли, вернитесь к нему.

---

## Часть 1: Теория (40 баллов)

Отвечайте своими словами, не копируйте.

### Вопрос 1 (8 баллов)

*(Слайды 5-6)*

а) Почему нельзя подать весь датасет в модель за один раз? Приведите конкретный пример с числами (размер данных vs память GPU).

б) Дайте определение трём понятиям: **Batch**, **Epoch**, **Iteration**. Для каждого приведите аналогию из жизни.

**Ваш ответ:**
а)

GPU имеет ограниченную память. Если датасет большой, он просто не поместится в память.

Пример:

- 60 000 изображений

- размер: 28×28×1

- тип float32 = 4 байта

Размер одного изображения 28 × 28 × 1 × 4 = 3136 байт ≈ 3 KB

Размер всего датасета 60 000 × 3 KB ≈ 180 MB

Но во время обучения нужно хранить ещё активации, градиенты и параметры модели.

В итоге может понадобиться гигабайты памяти, поэтому данные подают батчами.

б)

- Batch это часть датасета, которая подается в модель за один раз.

Аналогия:
как одна корзина товаров на кассе, а не весь магазин сразу.

- Epoch это один полный проход по всему датасету.

Аналогия:
прочитать всю книгу один раз.

- Iteration это один шаг обучения модели на одном батче.

Аналогия:
прочитать одну страницу книги.

### Вопрос 2 (8 баллов)

*(Слайды 9-10)*

Датасет содержит 60 000 образцов. Batch size = 128. Количество эпох = 15.

Посчитайте:
- Сколько итераций за одну эпоху?
- Сколько образцов останется в последнем (неполном) батче?
- Сколько итераций за все обучение?
- Если каждая итерация занимает 0.05 секунд, сколько МИНУТ займёт обучение?

**Ваш ответ:**
- Итерации за одну эпоху

60000 / 128 = 468.75 = 469 итераций

- Образцов в последнем батче

468 × 128 = 59904

60000 − 59904 = 96

- Итераций за всё обучение

469 × 15 = 7035

- Время обучения

7035 × 0.05 = 351.75 секунд ≈ 5.9 минут

### Вопрос 3 (8 баллов)

*(Слайд 13)*

Сравните маленький batch size (8) и большой (512). Для каждого назовите:
- Два преимущества
- Два недостатка

Какой batch size вы бы выбрали для обучения модели на GPU с 8 GB памяти на датасете из 100 000 картинок 128x128x3? Объясните выбор.

**Ваш ответ:**
- Маленький батч
  - Преимущества: меньше используется памяти GPU, иногда лучше обобщает модель
  - Недостатки: обучение медленнее, градиенты более шумные

- Большой batch

  - Преимущества: быстрее обучение, стабильнее градиенты

  - Недостатки: требует много памяти, может хуже обобщать

Для GPU 8 GB и датасета 100 000 изображений 128×128×3 я бы выбрала batch = 64 или 128

изображения достаточно большие, batch 512 может не поместиться в память

### Вопрос 4 (8 баллов)

*(Слайды 15, 18)*

а) Что такое Linear Scaling Rule? Вы обучали модель с batch=32 и lr=0.001. Хотите перейти на batch=256. Какой lr поставить и почему?

б) В чём разница между `model.train()` и `model.eval()`? Какие слои ведут себя по-разному и как именно?

**Ваш ответ:**

а) Linear Scaling Rule: если batch увеличили в k раз, то learning rate тоже увеличивают в k раз.

256 / 32 = 8

Новый learning rate: 0.001 × 8 = 0.008

б) Разница между model.train() и model.eval()

- model.train() это режим обучения, включается Dropout, BatchNorm обновляет статистику

- model.eval() это режим тестирования, Dropout отключается, BatchNorm использует сохранённые значения

Слои, которые ведут себя по-разному: Dropout и BatchNorm

### Вопрос 5 (8 баллов)

*(Слайды 17, 19-20)*

Найдите 4 ошибки в коде ниже и исправьте их:

```python
for epoch in range(10):
    for X_batch, y_batch in train_loader:
        output = model(X_batch)          # данные не на GPU
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()                 # забыли zero_grad
    
    # Оценка
    correct = 0
    total = 0
    for X_batch, y_batch in test_loader:
        output = model(X_batch.to(device))  # нет eval() и no_grad()
        preds = output.argmax(dim=1)
        correct += (preds == y_batch.to(device)).sum().item()
        total += len(y_batch)
```

Перепишите код правильно.

**Ваш ответ:**

for epoch in range(10):
    
    model.train()
    
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        output = model(X_batch)
        loss = criterion(output, y_batch)

        loss.backward()
        optimizer.step()
    
    
    # Оценка
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            output = model(X_batch)
            preds = output.argmax(dim=1)

            correct += (preds == y_batch).sum().item()
            total += len(y_batch)

---

## Часть 2: Код (60 баллов)

### Подготовка

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import time
import random

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Загружаем Fashion-MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])

train_data = datasets.FashionMNIST('./data', train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST('./data', train=False, download=True, transform=transform)

print(f'Train: {len(train_data)}')
print(f'Test:  {len(test_data)}')

class_names = ['Футболка', 'Брюки', 'Свитер', 'Платье', 'Пальто',
               'Сандалии', 'Рубашка', 'Кроссовки', 'Сумка', 'Ботинки']

100%|██████████| 26.4M/26.4M [00:01<00:00, 15.0MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 269kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.03MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 20.5MB/s]

Train: 60000
Test:  10000


### Задание 1: Создайте DataLoader (10 баллов)

*(Слайд 11)*

Создайте DataLoader для train и test данных.
- batch_size = 64
- shuffle = True для train, False для test
- drop_last = True для train

Выведите: количество батчей, размер одного батча.

In [ ]:
BATCH_SIZE = 64

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True
)

test_loader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Выведите количество батчей
print(f'Батчей в train: {len(train_loader)}')
print(f'Батчей в test:  {len(test_loader)}')

# Проверьте размер одного батча
for images, labels in train_loader:
    print(f'\nImages: {images.shape}')
    print(f'Labels: {labels.shape}')
    break

Батчей в train: 937
Батчей в test:  157

Images: torch.Size([64, 1, 28, 28])
Labels: torch.Size([64])


### Задание 2: Постройте модель (10 баллов)

*(Слайды 17-18)*

Создайте CNN для Fashion-MNIST:
- Conv2d(1, 16, 3, padding=1) + ReLU + MaxPool2d(2)
- Conv2d(16, 32, 3, padding=1) + ReLU + MaxPool2d(2)
- Flatten + Dropout(0.25) + Linear(32*7*7, 128) + ReLU + Linear(128, 10)

In [ ]:
class FashionCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)

        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)

        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(0.25)

        self.fc1 = nn.Linear(1568, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):

        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))

        x = self.flatten(x)
        x = self.dropout(x)

        x = self.relu(self.fc1(x))
        x = self.fc2(x)

        return x


model = FashionCNN().to(device)

print(model)
print(f'\nПараметров: {sum(p.numel() for p in model.parameters()):,}')

FashionCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu): ReLU()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (dropout): Dropout(p=0.25, inplace=False)
  (fc1): Linear(in_features=1568, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)

Параметров: 206,922


### Задание 3: Напишите Training Loop (15 баллов)

*(Слайды 17, 19)*

Напишите полный цикл обучения на 10 эпох:
- CrossEntropyLoss, Adam с lr=0.001
- Сохраняйте train_loss и test_accuracy на каждой эпохе
- Не забудьте: model.train(), model.eval(), .to(device), torch.no_grad()
- Выводите прогресс каждые 2 эпохи

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

NUM_EPOCHS = 10

train_losses = []
test_accs = []

for epoch in range(NUM_EPOCHS):

    # ---- TRAIN ----
    model.train()

    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    train_losses.append(epoch_loss)


    # ---- TEST ----
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    accuracy = correct / total
    test_accs.append(accuracy)


    if (epoch + 1) % 2 == 0:
        print(f'Epoch {epoch+1}/{NUM_EPOCHS}  '
              f'Loss: {train_losses[-1]:.4f}  '
              f'Acc: {test_accs[-1]:.2%}')

Epoch 2/10  Loss: 0.3208  Acc: 88.81%
Epoch 4/10  Loss: 0.2462  Acc: 88.38%
Epoch 6/10  Loss: 0.2105  Acc: 90.87%
Epoch 8/10  Loss: 0.1822  Acc: 91.86%


### Задание 4: Графики обучения (5 баллов)

Постройте два графика рядом: Train Loss и Test Accuracy по эпохам.

In [ ]:
plt.figure(figsize=(12,5))

# Train Loss
plt.subplot(1,2,1)
plt.plot(train_losses)
plt.title("Train Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")

# Test Accuracy
plt.subplot(1,2,2)
plt.plot(test_accs)
plt.title("Test Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.show()



### Задание 5: Эксперимент с Batch Size (15 баллов)

*(Слайды 13-14)*

Обучите ту же модель FashionCNN с тремя разными batch size: **16**, **64**, **256**. Для каждого:
- Обучите на 10 эпох (lr=0.001 для всех)
- Засеките время обучения
- Запишите финальную accuracy

Постройте:
1. График Test Accuracy по эпохам (все три на одном графике)
2. Барчарт времени обучения

Напишите вывод: какой batch size лучше и почему?

In [ ]:
def train_model(batch_size, lr=0.001, num_epochs=10):

    train_loader = DataLoader(
        train_data,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True
    )

    test_loader = DataLoader(
        test_data,
        batch_size=batch_size,
        shuffle=False
    )

    model = FashionCNN().to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    train_losses = []
    test_accs = []

    start_time = time.time()

    for epoch in range(num_epochs):

        # TRAIN
        model.train()
        running_loss = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / len(train_loader)
        train_losses.append(epoch_loss)

        # TEST
        model.eval()

        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in test_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                preds = outputs.argmax(dim=1)

                correct += (preds == labels).sum().item()
                total += labels.size(0)

        acc = correct / total
        test_accs.append(acc)

    end_time = time.time()

    training_time = end_time - start_time

    return train_losses, test_accs, training_time

batch_sizes = [16, 64, 256]

results = {}

for bs in batch_sizes:

    print(f"\nTraining with batch size = {bs}")

    losses, accs, t = train_model(bs)

    results[bs] = {
        "losses": losses,
        "accs": accs,
        "time": t
    }

    print(f"Final accuracy: {accs[-1]:.2%}")
    print(f"Training time: {t:.2f} sec")

In [ ]:
plt.figure(figsize=(8,5))

for bs in batch_sizes:
    plt.plot(results[bs]["accs"], label=f"Batch {bs}")

plt.title("Test Accuracy by Epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.show()

**Ваш вывод:**

Batch size сильно влияет на скорость обучения и точность модели.
Маленький batch (16) обучается медленнее, потому что выполняется больше итераций.
Большой batch (256) обучается быстрее, но иногда даёт немного худшую обобщающую способность.

В эксперименте batch size = 64 показал лучший баланс между скоростью обучения и точностью модели.

Поэтому batch 64 является оптимальным выбором для данного датасета и архитектуры.

### Задание 6: Визуализация предсказаний (5 баллов)

Возьмите лучшую модель из Задания 5. Покажите 16 случайных картинок из тестового набора с предсказаниями модели. Правильные -- зелёным, неправильные -- красным.

In [ ]:
model.eval()

indices = random.sample(range(len(test_data)), 16)

plt.figure(figsize=(8,8))

with torch.no_grad():

    for i, idx in enumerate(indices):

        image, label = test_data[idx]

        input_img = image.unsqueeze(0).to(device)

        output = model(input_img)
        pred = output.argmax(dim=1).item()

        correct = (pred == label)

        color = "green" if correct else "red"

        plt.subplot(4,4,i+1)

        plt.imshow(image.squeeze(), cmap="gray")

        plt.title(
            f"P: {class_names[pred]}\nT: {class_names[label]}",
            color=color,
            fontsize=9
        )

        plt.axis("off")

plt.tight_layout()
plt.show()


---

## Часть 3: Бонус (20 баллов)

### Бонус 1: Linear Scaling Rule на Fashion-MNIST (10 баллов)

Обучите модель с тремя конфигурациями:
1. batch=32, lr=0.001 (baseline)
2. batch=128, lr=0.001 (без коррекции)
3. batch=128, lr=0.004 (с linear scaling: batch x4 -> lr x4)

Постройте график accuracy и напишите вывод: помогает ли Linear Scaling Rule?

In [ ]:
# Ваш код здесь



**Ваш вывод:**



### Бонус 2: Переобучение (10 баллов)

Обучите модель на **маленькой части** Fashion-MNIST (используйте только первые 500 образцов из train). Обучите на 50 эпох.

Постройте на одном графике Train Loss и Test Loss по эпохам. Видите ли вы переобучение? Когда оно начинается? Какой batch size и сколько эпох были бы оптимальны?

In [ ]:
# Ваш код здесь



**Ваш вывод:**

